# 📰 Fake News Detection using Graph ML
## Notebook 2: Graph Construction + Feature Engineering

**What we build here:**
- A **heterogeneous graph** where:
  - **Nodes** = News articles, Speakers, Subjects
  - **Edges** = Speaker made statement, Statement belongs to subject
- Node features using **TF-IDF** and one-hot encodings
- Save graph as `PyTorch Geometric` Data object

```
[Speaker] ──spoke──▶ [Article] ──about──▶ [Subject]
```

## Step 1 — Install Libraries

In [ ]:
!pip install torch torchvision torchaudio -q
!pip install torch-geometric -q
!pip install networkx matplotlib scikit-learn pandas -q
print('✅ Done installing!')

In [ ]:
import pandas as pd
import numpy as np
import torch
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from torch_geometric.data import Data
import os, pickle, warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Libraries loaded | Device: {device}')

## Step 2 — Load Processed Data

In [ ]:
train_df = pd.read_csv('data/train.csv')
val_df   = pd.read_csv('data/val.csv')
test_df  = pd.read_csv('data/test.csv')

# Combine all splits — graph is built on all data, masks separate train/val/test
df = pd.concat([train_df, val_df, test_df], ignore_index=True)
df['binary_label'] = df['binary_label'].fillna(1).astype(int)
df['speaker']  = df['speaker'].fillna('unknown')
df['subject']  = df['subject'].fillna('unknown')
df['statement'] = df['statement'].fillna('')

print(f'Total statements: {len(df):,}')
df.head(3)

## Step 3 — Build the Graph

### Node types
| Type | Description | Feature |
|------|-------------|------|
| Article | Each news statement | TF-IDF (500-dim) |
| Speaker | Person who made the claim | One-hot encoded |
| Subject | Topic of the statement | One-hot encoded |

### Edge types
| Type | Meaning |
|------|---------|
| speaker→article | This speaker made this statement |
| article→subject | This statement is about this subject |

In [ ]:
# ── Encode speakers and subjects ──────────────────────────────────────────────
speaker_enc = LabelEncoder()
subject_enc = LabelEncoder()
df['speaker_id'] = speaker_enc.fit_transform(df['speaker'])
df['subject_id'] = subject_enc.fit_transform(df['subject'])

n_articles = len(df)
n_speakers = df['speaker_id'].nunique()
n_subjects = df['subject_id'].nunique()

print(f'Nodes — Articles: {n_articles:,} | Speakers: {n_speakers:,} | Subjects: {n_subjects:,}')
print(f'Total nodes     : {n_articles + n_speakers + n_subjects:,}')

In [ ]:
# ── Node features ─────────────────────────────────────────────────────────────
# Article features: TF-IDF on statement text (500 features)
tfidf = TfidfVectorizer(max_features=500, stop_words='english', ngram_range=(1, 2))
article_feats = tfidf.fit_transform(df['statement']).toarray().astype(np.float32)  # (N_art, 500)

# Speaker features: one-hot (n_speakers, n_speakers) — identity matrix
speaker_feats = np.eye(n_speakers, dtype=np.float32)  # (N_spk, N_spk)

# Subject features: one-hot
subject_feats = np.eye(n_subjects, dtype=np.float32)  # (N_sub, N_sub)

# Pad all features to the same dim (max of 500, n_speakers, n_subjects)
feat_dim = max(500, n_speakers, n_subjects)

def pad(arr, dim):
    pad_width = dim - arr.shape[1]
    if pad_width > 0:
        arr = np.hstack([arr, np.zeros((arr.shape[0], pad_width), dtype=np.float32)])
    return arr

article_feats = pad(article_feats, feat_dim)
speaker_feats = pad(speaker_feats, feat_dim)
subject_feats = pad(subject_feats, feat_dim)

# Concatenate all node features: [articles | speakers | subjects]
all_feats = np.vstack([article_feats, speaker_feats, subject_feats])  # (N_total, feat_dim)
x = torch.tensor(all_feats, dtype=torch.float)

print(f'Node feature matrix shape: {x.shape}  (total_nodes x feat_dim)')

In [ ]:
# ── Build edge list ────────────────────────────────────────────────────────────
# Node ID offsets
spk_offset = n_articles
sub_offset = n_articles + n_speakers

src_edges, dst_edges = [], []

for idx, row in df.iterrows():
    art_node = idx
    spk_node = spk_offset + row['speaker_id']
    sub_node = sub_offset + row['subject_id']

    # Speaker ↔ Article (bidirectional)
    src_edges += [spk_node, art_node]
    dst_edges += [art_node, spk_node]

    # Article ↔ Subject (bidirectional)
    src_edges += [art_node, sub_node]
    dst_edges += [sub_node, art_node]

edge_index = torch.tensor([src_edges, dst_edges], dtype=torch.long)
print(f'Edge index shape: {edge_index.shape}  ({edge_index.shape[1]:,} edges total)')

In [ ]:
# ── Labels and masks (article nodes only) ──────────────────────────────────────
y = torch.tensor(df['binary_label'].values, dtype=torch.long)

n_train = len(train_df)
n_val   = len(val_df)
n_test  = len(test_df)

train_mask = torch.zeros(len(df), dtype=torch.bool)
val_mask   = torch.zeros(len(df), dtype=torch.bool)
test_mask  = torch.zeros(len(df), dtype=torch.bool)

train_mask[:n_train] = True
val_mask[n_train:n_train+n_val] = True
test_mask[n_train+n_val:] = True

print(f'Labels shape : {y.shape}')
print(f'Train mask   : {train_mask.sum().item():,} nodes')
print(f'Val mask     : {val_mask.sum().item():,} nodes')
print(f'Test mask    : {test_mask.sum().item():,} nodes')

In [ ]:
# ── Assemble PyG Data object ───────────────────────────────────────────────────
# We only predict on article nodes (first n_articles nodes)
# Extend labels + masks to cover all nodes (speaker/subject nodes get label=-1)
full_y = torch.full((x.shape[0],), -1, dtype=torch.long)
full_y[:n_articles] = y

full_train = torch.zeros(x.shape[0], dtype=torch.bool)
full_val   = torch.zeros(x.shape[0], dtype=torch.bool)
full_test  = torch.zeros(x.shape[0], dtype=torch.bool)
full_train[:n_articles] = train_mask
full_val[:n_articles]   = val_mask
full_test[:n_articles]  = test_mask

graph_data = Data(
    x          = x,
    edge_index = edge_index,
    y          = full_y,
    train_mask = full_train,
    val_mask   = full_val,
    test_mask  = full_test,
    num_classes= 2
)

print('\n✅ PyTorch Geometric Data object:')
print(graph_data)
print(f'\nIs undirected: {graph_data.is_undirected()}')

## Step 4 — Visualize a Sample of the Graph

In [ ]:
# Build a small NetworkX subgraph for visualization (first 50 articles + their neighbors)
G = nx.Graph()

SAMPLE = 50  # number of article nodes to visualize
sample_idx = list(range(SAMPLE))

node_colors = []
node_sizes  = []
node_labels = {}

# Add article nodes
for i in sample_idx:
    label = df.iloc[i]['binary_label']
    G.add_node(i, node_type='article', label=label)
    node_colors.append('#e74c3c' if label == 1 else '#2ecc71')
    node_sizes.append(120)
    node_labels[i] = ''

# Add speaker and subject nodes connected to sample articles
added_speakers = {}
added_subjects = {}

for i in sample_idx:
    row = df.iloc[i]
    spk = f"spk_{row['speaker_id']}"
    sub = f"sub_{row['subject_id']}"

    if spk not in G:
        G.add_node(spk, node_type='speaker')
        node_colors.append('#3498db')
        node_sizes.append(80)
        node_labels[spk] = ''

    if sub not in G:
        G.add_node(sub, node_type='subject')
        node_colors.append('#f39c12')
        node_sizes.append(80)
        node_labels[sub] = ''

    G.add_edge(i, spk)
    G.add_edge(i, sub)

print(f'Sample graph — Nodes: {G.number_of_nodes()} | Edges: {G.number_of_edges()}')

# Draw
plt.figure(figsize=(14, 10))
pos = nx.spring_layout(G, k=0.6, seed=42)
nx.draw_networkx(G, pos,
                 node_color=node_colors,
                 node_size=node_sizes,
                 labels=node_labels,
                 edge_color='#cccccc',
                 width=0.5,
                 alpha=0.85)

legend = [
    mpatches.Patch(color='#e74c3c', label='Fake article'),
    mpatches.Patch(color='#2ecc71', label='Real article'),
    mpatches.Patch(color='#3498db', label='Speaker'),
    mpatches.Patch(color='#f39c12', label='Subject'),
]
plt.legend(handles=legend, loc='upper left', fontsize=11)
plt.title(f'Knowledge Graph — {SAMPLE} articles + their speakers & subjects', fontsize=14)
plt.axis('off')
plt.tight_layout()
plt.savefig('graph_visualization.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Graph saved as graph_visualization.png')

## Step 5 — Save Graph for Training

In [ ]:
os.makedirs('data', exist_ok=True)
torch.save(graph_data, 'data/graph_data.pt')

# Save metadata
meta = {
    'n_articles': n_articles,
    'n_speakers': n_speakers,
    'n_subjects': n_subjects,
    'feat_dim'  : feat_dim,
    'n_train'   : n_train,
    'n_val'     : n_val,
    'n_test'    : n_test,
}
with open('data/meta.pkl', 'wb') as f:
    pickle.dump(meta, f)

print('✅ Graph saved to data/graph_data.pt')
print('✅ Metadata saved to data/meta.pkl')
print(f'\nGraph summary:')
print(f'  Total nodes : {graph_data.num_nodes:,}')
print(f'  Total edges : {graph_data.num_edges:,}')
print(f'  Node features: {graph_data.num_node_features}')
print('\n🚀 Ready for Notebook 3: GNN Training!')